# Aprendiendo con pocos datos
Óscar Niño Robles

## Referencias

ProtoNet: Snell, J. et al. Prototypical networks for few-shot learning. Se puede consultar el PDF [aquí](https://arxiv.org/pdf/1703.05175)

## Idea intuitiva

Vamos a recordar la idea intuitiva de esté método. Este método consiste en entrenar un encoder con clases que no son la que queremos clasificar (de hecho tienen que ser **clases en las que no tengamos falta de datos**) en lugar de usar una red neuronal para clasificar.


Para clasificar la idea es clasificar como la clase que esté más cercana, como una especie de *clustering*. Es decir, con los datos de entrenamiento nos aseguramos de tener un buen *encoder*, luego pasamos las clases de las que tenemos pocos datos y para predecir basta con medir distancias al punto central (el promedio por cada clase) de cada clase. Esto hace que la red no aprenda a clasificar, por lo que no pasa nada al tener pocos datos porque la red no tiene que extraer características, simplemente ver si lo que queremos predecir se parece más a esa clase que no a otra.


Este método en general da muy buenos resultados, la única "pega" que puede tener es que sí que necesitamos bastante datos similares a los que queremos precedir para tener un buen encoder. Por ejemplo si queremos clasificar ciertos animales necesitamos datos sobre otros animales, que no sean los que queremos clasificar (que de esos tendremos pocos datos).

Por ejemplo en nuestra implementación, para las pruebas usaremos 5 imágenes de cada clase, es decir, **simularemos que tenemos solo 5 imágenes de las clases**

## Importaciones

Importamos las librerias necesarias

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

Vemos la versión de tensorflow

In [2]:
import tensorflow as tf
print(tf.__version__)

2.20.0


## Ejecución con CIFAR-10

### Creación del encoder

Como el código lo vamos a desarollar desde 0, lo único "especial" que tenemos que hacer para adaptar a este dataset es **definir el tamaño de las imágenes de input como 32x32**

Lo primero que vamos a hacer es definir una función que cree el encoder (la red neuronal que vamos a entrenar) que la haremos tal y como dice el paper, con 4 bloques convolucionales iguales, usando Batch Normalization con 64 filtros, ReLU  y max pooling. Añadimos también *data augmentation*.

In [3]:
def crear_encoder():
    """
    Crea la arquitectura de embedding f_phi descrita en el paper.
    """

    modelo = models.Sequential(name="ProtoNet_Encoder")

    # --- DATA AUGMENTATION  ---
    # Voltea horizontalmente al azar y desplaza la imagen un 10%
    modelo.add(layers.RandomFlip("horizontal", input_shape=(32, 32, 3))) #Tamaño de CIFAR-10
    modelo.add(layers.RandomTranslation(height_factor=0.1, width_factor=0.1))


    # El paper indica 4 bloques convolucionales idénticos
    for i in range(4):
        # 1. Convolución de 64 filtros (3x3)
        # Usamos padding='same' para que la reducción de tamaño la dicte solo el MaxPool
        modelo.add(layers.Conv2D(64, (3, 3), padding='same', input_shape=(32, 32, 3) if i == 0 else None))

        # 2. Batch Normalization
        modelo.add(layers.BatchNormalization())

        # 3. ReLU nonlinearity
        modelo.add(layers.ReLU())

        # 4. Max-pooling (2x2)
        modelo.add(layers.MaxPool2D((2, 2)))

    # Finalmente aplanamos para obtener nuestro vector de embedding (c_k o f_phi(x))
    modelo.add(layers.Flatten())

    return modelo

# Instanciamos el modelo
encoder = crear_encoder()

# Veamos la estructura y confirmemos el tamaño de salida
encoder.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "ProtoNet_Encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation              │ (None, 32, 32, 3)      │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 32, 32, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 4, 4, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 4, 4, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 2, 2, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 256)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 113,600 (443.75 KB)

 Trainable params: 113,088 (441.75 KB)

 Non-trainable params: 512 (2.00 KB)

Vemos que finalmente nos estaríamos quedando con vectores de 256 dimensiones (2x2x64=256). Es decir estaremos trabajando en un espacio de 256 dimensiones

### Procesamiento de datos

Vamos a cargar el dataset CIFAR-10, también **separaremos en clases de entrenamiento (7 clases) y en clase de test (3 clases)**. Recordemos que las clases de test tienen que ser completamente nuevas para el modelo, se usarán las clases de train para entrenar bien el encoder.

Tomamos 7 clases de entrenamiento y 3 de test porque en el propio paper se demuestra que **el modelo funciona mejor entrenándolo con más clases de las que luego va a tener que clasificar.**

In [4]:
# ==========================================
#  CARGA Y PREPROCESAMIENTO DE CIFAR-10
# ==========================================
print("Cargando CIFAR-10...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalizamos las imágenes para que los píxeles vayan de 0 a 1 (vital para la red neuronal)
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# ==========================================
#  SEPARACIÓN DE CLASES
# ==========================================
# En Few-Shot, el meta-entrenamiento usa clases distintas a las pruebas.
# Vamos a usar las clases del 0 al 6 (aviones, coches, pájaros, gatos, ciervos, perros,ranas) para entrenar.
clases_entrenamiento = [0, 1, 2, 3, 4, 5,6]

# Filtramos el dataset original para quedarnos SOLO con esas 6 clases
mascara_train = np.isin(y_train, clases_entrenamiento).flatten()
x_train_fewshot = x_train[mascara_train]
y_train_fewshot = y_train[mascara_train].flatten()

# Las clases del 7 al 9 (caballos, barcos, camiones) las guardaremos
# para evaluar el modelo al final (nuestro "mundo real").
clases_test = [7, 8, 9]
mascara_test = np.isin(y_test, clases_test).flatten()
x_test_fewshot = x_test[mascara_test]
y_test_fewshot = y_test[mascara_test].flatten()

print(f"Imágenes para Meta-Entrenamiento: {len(x_train_fewshot)}")

Cargando CIFAR-10...
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Imágenes para Meta-Entrenamiento: 35000


### Preparativos para el entrenamiento: Generación de episodios y cálculo de métricas

Vemos ahora cómo se generan los episodios de entrenamiento (Algoritmo 1 del paper). Para ellos hay que ir eligiendo clases y de esas clases el *support set* (las imagénes que se usarán de referencia) y el *query set* (las imágenes que se usarán para obtener predicciones)

In [5]:
def generar_episodio(x, y, n_classes, n_support, n_query):
    """
    Implementación del Algoritmo 1 del paper para formar episodios.
    """
    # Paso 1: Elegir clases al azar para este episodio (V en el Algoritmo 1 del paper)
    clases_unicas = np.unique(y)
    clases_seleccionadas = np.random.choice(clases_unicas, size=n_classes, replace=False)

    support_set = []
    query_set = []

    # Paso 2: Iterar sobre cada clase seleccionada
    for c in clases_seleccionadas:
        # Filtrar imágenes para obtener solo las de la clase 'c' (D_k en el paper)
        x_clase = x[y == c]

        # Muestreo aleatorio sin reemplazo (RANDOMSAMPLE)
        indices = np.random.choice(len(x_clase), size=n_support + n_query, replace=False)

        # Paso 3: Separar en soporte (S_k) y consulta (Q_k)
        soporte = x_clase[indices[:n_support]]
        consulta = x_clase[indices[n_support:]]

        support_set.append(soporte)
        query_set.append(consulta)

    # Convertimos las listas a tensores de TensorFlow
    # Las dimensiones finales serán: [n_classes, n_support_o_query, alto, ancho, canales]
    tensor_support = tf.convert_to_tensor(support_set, dtype=tf.float32)
    tensor_query = tf.convert_to_tensor(query_set, dtype=tf.float32)

    return tensor_support, tensor_query

Ahora calculamos el *accuracy* y la *loss* que se han obtenido en cada episodio usando el *query set* (las imágenes que se han predicho a ver si las predicciones son correctas o no).

In [6]:
def calcular_perdida_y_precision(support, query, encoder):
    # support shape original: [n_classes, n_support, 32, 32, 3]
    # query shape original: [n_classes, n_query, 32, 32, 3]

    n_classes = tf.shape(support)[0]
    n_support = tf.shape(support)[1]
    n_query = tf.shape(query)[1]

    # -----------------------------------------------------------
    # PASO 1: Extraer Embeddings
    # -----------------------------------------------------------
    # Keras espera un bloque plano [Lote, 32, 32, 3], así que "aplastamos"
    # las dimensiones de clase y shot para meterlas en el encoder.
    support_plano = tf.reshape(support, [-1, 32, 32, 3])
    query_plano = tf.reshape(query, [-1, 32, 32, 3])

    # Pasamos las imágenes por la red (importante poner training=True)
    z_support = encoder(support_plano, training=True) # Sale: [n_classes * n_support, 256]
    z_query = encoder(query_plano, training=True)     # Sale: [n_classes * n_query, 256]

    # -----------------------------------------------------------
    # PASO 2: Calcular Prototipos
    # -----------------------------------------------------------
    # Devolvemos a z_support su forma separada por clases: [n_classes, n_support, 256]
    embedding_dim = tf.shape(z_support)[-1]
    z_support_agrupado = tf.reshape(z_support, [n_classes, n_support, embedding_dim])

    # El prototipo es la media: Hacemos la media en el eje 1 (el eje de los n_support)
    # Resultado: Un vector promedio por clase -> [n_classes, 256]
    prototipos = tf.reduce_mean(z_support_agrupado, axis=1)

    # -----------------------------------------------------------
    # PASO 3: Calcular Distancias Euclidianas
    # -----------------------------------------------------------
    # Queremos comparar CADA query (ej. 15 queries) contra CADA prototipo (ej. 5 clases).
    # Usamos un "truco" de TensorFlow llamado Broadcasting expandiendo dimensiones:

    # z_query expandido: [n_classes * n_query, 1, 256]
    z_query_exp = tf.expand_dims(z_query, axis=1)
    # prototipos expandidos: [1, n_classes, 256]
    prototipos_exp = tf.expand_dims(prototipos, axis=0)

    # Distancia Euclídea al cuadrado: sumatoria de (punto A - punto B)^2
    # El resultado final es una matriz de distancias de tamaño: [n_classes * n_query, n_classes]
    distancias = tf.reduce_sum(tf.square(z_query_exp - prototipos_exp), axis=2)

    # -----------------------------------------------------------
    # PASO 4: Calcular la Loss y la Precisión
    # -----------------------------------------------------------
    # El paper dice que apliquemos un Softmax sobre el NEGATIVO de las distancias.
    # ¿Por qué negativo? Porque Softmax da más probabilidad a los números más grandes.
    # Una distancia pequeña (cerca) se convierte en un número grande (alta probabilidad).
    logits = -distancias

    # Generamos las etiquetas reales. Como metimos los queries en orden, las etiquetas
    # son simplemente [0,0,0..., 1,1,1..., 2,2,2...]
    etiquetas = tf.repeat(tf.range(n_classes), n_query)

    # TensorFlow calcula el Softmax y la pérdida por entropía cruzada todo junto aquí:
    loss = tf.keras.losses.sparse_categorical_crossentropy(etiquetas, logits, from_logits=True)
    loss = tf.reduce_mean(loss)

    # Comprobamos la precisión (Accuracy) para ver si va aprendiendo
    predicciones = tf.argmax(logits, axis=1, output_type=tf.int32)
    accuracy = tf.reduce_mean(tf.cast(tf.equal(predicciones, etiquetas), tf.float32))

    return loss, accuracy

### Entrenamiento y evaluación

Vamos a entrenar el modelo, para ello basta definir los parámetros, y hacer un bucle donde por cada episodio genere las imágenes calculen la pérdida y el *accuracy*, el modelo vaya aprendiento y se vaya imprimiendo por pantalla cómo va el proceso.

**Nota:** A pesar de que el paper indique que es mejor entrenar usando el *shot* que vamos a utilizar luego para predecir, en nuestro caso por eficiencia computacional, para no entrenar dos *encoders* vamos a entrenar solo uno con *5 shot* y ese mismo lo vamos a usar tanto para medir *5 shot* como *1 shot* en el test. Reiterando que esto no es lo ideal, pero a nivel académico no aporta nada realizar este entrenamiento pues sería igual que el que haremos a continuación pero sustituyendo 1 por 5.

In [7]:
# Variables antes del bucle para guardar el historial reciente
historico_loss = []
historico_acc = []

# Definimos el optimizador siguiendo el paper (Adam con learning rate inicial de 0.001)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# En Few-Shot Learning, a menudo llamamos "épocas" directamente a los episodios
episodios_entrenamiento = 3500

# Configuramos la forma de los entrenamientos (números fijos)
# Vamos a entrenar con 5 clases por episodio (de las 7 disponibles elige 5 en cada episodio)
# (5-way), 5 fotos de soporte (5-shot) y 15 queries.
n_clases_train = 5
n_support_train = 5
n_query_train = 15

print(" Iniciando el Entrenamiento de ProtoNets...")

for episodio in range(episodios_entrenamiento):

    # GENERAR LOS DATOS DEL EPISODIO
    # (x_train_fewshot e y_train_fewshot son las imágenes de nuestras clases base)
    support_set, query_set = generar_episodio(
        x_train_fewshot,
        y_train_fewshot,
        n_classes=n_clases_train,
        n_support=n_support_train,
        n_query=n_query_train
    )

    # ENCENDER LA "GRABADORA" DE GRADIENTES
    with tf.GradientTape() as tape:
        # HACER EL FORWARD PASS (Pasar imágenes y calcular matemáticas)
        loss, accuracy = calcular_perdida_y_precision(support_set, query_set, encoder)

    # CALCULAR EL ERROR Y LOS GRADIENTES
    # Le decimos a TF: "¿Cómo afectan los pesos del 'encoder' a la 'loss' final?"
    gradientes = tape.gradient(loss, encoder.trainable_variables)

    # ACTUALIZAR LOS PESOS (Aprender)
    optimizer.apply_gradients(zip(gradientes, encoder.trainable_variables))

   # Guardamos los resultados de este episodio en la lista
    historico_loss.append(loss.numpy())
    historico_acc.append(accuracy.numpy())

    # Mostrar progreso promediando los últimos 100 episodios
    if (episodio + 1) % 100 == 0:
        media_loss = np.mean(historico_loss[-100:])
        media_acc = np.mean(historico_acc[-100:])
        print(f"Episodios {episodio-99} a {episodio + 1} | Loss Media: {media_loss:.4f} | Accuracy Media: {media_acc*100:.2f}%")

print("¡Entrenamiento completado! El espacio de embedding ha sido creado.")

 Iniciando el Entrenamiento de ProtoNets...
Episodios 0 a 100 | Loss Media: 2.8880 | Accuracy Media: 41.11%
Episodios 100 a 200 | Loss Media: 1.1874 | Accuracy Media: 49.15%
Episodios 200 a 300 | Loss Media: 1.1793 | Accuracy Media: 49.81%
Episodios 300 a 400 | Loss Media: 1.1060 | Accuracy Media: 53.60%
Episodios 400 a 500 | Loss Media: 1.1049 | Accuracy Media: 53.23%
Episodios 500 a 600 | Loss Media: 1.0653 | Accuracy Media: 55.05%
Episodios 600 a 700 | Loss Media: 1.0521 | Accuracy Media: 56.39%
Episodios 700 a 800 | Loss Media: 1.0538 | Accuracy Media: 56.15%
Episodios 800 a 900 | Loss Media: 1.0277 | Accuracy Media: 58.40%
Episodios 900 a 1000 | Loss Media: 0.9870 | Accuracy Media: 59.72%
Episodios 1000 a 1100 | Loss Media: 0.9927 | Accuracy Media: 58.97%
Episodios 1100 a 1200 | Loss Media: 0.9284 | Accuracy Media: 61.49%
Episodios 1200 a 1300 | Loss Media: 0.9272 | Accuracy Media: 61.83%
Episodios 1300 a 1400 | Loss Media: 0.9433 | Accuracy Media: 61.79%
Episodios 1400 a 1500 | L

Por último vamos a evaluar el modelo creado con los datos de test. Como en cada episodio se toman tan pocas imágenes, aprevechando que tenemos bastantes podemos hacer varios episodios y promediar para obtener así una métrica más exacta, es decir, es como si hicieramos "varios exámenes".

Recordemos que el interés de esto es trabajar con **pocos datos**, así que bajo este contexto en el "mundo real", esto no funcionaría así. Pero en nuestra experimentación para ver si el modelo funciona es ideal.

In [8]:
def evaluar_modelo(x_test_base, y_test_base, encoder, n_vias=3, n_soporte=5, n_consultas=15, n_episodios=100):
    """
    Evalúa el rendimiento del modelo en clases que nunca vio.
    """
    precisiones_totales = []

    print(f"Evaluando en {n_episodios} episodios de prueba...")

    for i in range(n_episodios):
        # Generar un episodio de prueba con las clases reservadas (ej. Ranas, Caballos...)
        soporte, consultas = generar_episodio(
            x_test_base,
            y_test_base,
            n_classes=n_vias,
            n_support=n_soporte,
            n_query=n_consultas
        )

        # Inferencia: Calculamos pérdida y precisión (sin GradientTape, no queremos aprender)
        # Usamos training=False porque el BatchNormalization debe usar sus valores aprendidos
        loss, accuracy = calcular_perdida_y_precision(soporte, consultas, encoder)

        precisiones_totales.append(accuracy.numpy())

    # Calculamos la media final de todos los episodios
    precision_media = np.mean(precisiones_totales)
    desviacion_tipica = np.std(precisiones_totales)

    print(f"Precisión final en clases nuevas: {precision_media*100:.2f}% ± {desviacion_tipica*100:.2f}%")

    # Devolvemos ambas variables por si quieres usarlas en variables fuera de la función
    return precision_media, desviacion_tipica


In [9]:
# Ejecutamos la evaluación final 5 shot
# Usamos nuestras clases de CIFAR-10 que separamos al principio (6 al 9)
evaluar_modelo(x_test_fewshot, y_test_fewshot, encoder)

Evaluando en 100 episodios de prueba...
Precisión final en clases nuevas: 75.69% ± 7.65%


(np.float32(0.756889), np.float32(0.076453485))

Vemos que el modelo funciona muy bien, partiendo de 5 imagenes de entrenamiento solo y evaluando 15 imágenes hay una precisión media de un 75.69%, con una desviación típica de 7.65%

Vamos ahora a evaluar el modelo si fuera *1 shot*, que recordamos que siguiendo el paper deberíamos de usar un *encoder* entrenado de forma *1 shot* pero por eficiencia computacional vamos a usar el mismo.

In [10]:
evaluar_modelo(x_test_fewshot, y_test_fewshot, encoder,n_soporte=1)

Evaluando en 100 episodios de prueba...
Precisión final en clases nuevas: 62.36% ± 10.73%


(np.float32(0.62355554), np.float32(0.10725854))

Vemos que alcanza un resultado de un 62.36% con una desviación típica de 10.73%, que aunque no es el impresionante 75.69% que alcanzamos antes recordamos que el modelo ahora solo está recibiendo 1 ejemplo para su tarea de clasificación. Lo cuál es impresionante, teniendo en cuenta que el puro azar daría como resultado un *accuracy* de 33%.

Si hubieramos entrenado el encoder con *1 shot* el resultado hubiera sido ligeramente mejor.

Recogemos los resultados en una tabla

#### Tabla 1: Resultados Few-Shot Learning

| Modelo (Arquitectura) | Métrica de Distancia | 3-way 1-shot | 3-way 5-shot |
| :--- | :---: | :---: | :---: |
| Puro Azar (Baseline) | - | 33.33% | 33.33% |
| **Prototypical Networks (Nuestra Impl.)** | **Euclidiana** | **62.36% ± 10.73%** | **75.69% ± 7.65%** |


### Zero shot

Si antes teníamos pocas imágenes de las clases a predecir, ahora no tenemos **ninguna**. Lo que tenemos es sólo un vector que define a la clase con atributos. El objetivo es a partir de ahí aprender siguiendo la misma idea que antes.

Por un lado tenemos un encoder de imágenes ya entrenados, por otro tenemos que entrenar otro para pasar de los vectores atributos al mismo espacio de 256 dimensiones donde viven las imágenes tras pasarlas por el encoder entrenado. Esto se hace para unificar las distan

In [ ]:

# ==========================================
#  MATRIZ DE ATRIBUTOS
# ==========================================
# Atributos: [Ruedas, Pelo, Alas, Motor, Ojos, Cuernos, Doméstico, Felino, Pezuñas, Pesado]
matriz_atributos = np.array([
    [1, 0, 1, 1, 0, 0, 0, 0, 0, 1], # 0: Avión   (Ruedas, Alas, Motor, Pesado)
    [1, 0, 0, 1, 0, 0, 0, 0, 0, 0], # 1: Coche   (Ruedas, Motor)
    [0, 0, 1, 0, 1, 0, 0, 0, 0, 0], # 2: Pájaro  (Alas, Ojos)
    [0, 1, 0, 0, 1, 0, 1, 1, 0, 0], # 3: Gato    (Pelo, Ojos, Doméstico, Felino)
    [0, 1, 0, 0, 1, 1, 0, 0, 1, 0], # 4: Ciervo  (Pelo, Ojos, Cuernos, Pezuñas)
    [0, 1, 0, 0, 1, 0, 1, 0, 0, 0], # 5: Perro   (Pelo, Ojos, Doméstico)
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0], # 6: Rana    (Ojos)
    [0, 1, 0, 0, 1, 0, 1, 0, 1, 1], # 7: Caballo (Pelo, Ojos, Doméstico, Pezuñas, Pesado)
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 1], # 8: Barco   (Motor, Pesado)
    [1, 0, 0, 1, 0, 0, 0, 0, 0, 1]  # 9: Camión  (Ruedas, Motor, Pesado)
], dtype='float32')

# ==========================================
# ENCODER DE ATRIBUTOS
# ==========================================
def crear_attribute_encoder():
    modelo = models.Sequential([
        # La entrada es de 10 dimensiones
        layers.Input(shape=(10,)),
        layers.Dense(64, activation='relu'),
        # Lo expandimos a 256 para que encaje con el encoder de imágenes
        layers.Dense(256)
    ], name="Attribute_Encoder")
    return modelo

# Creamos las dos redes (usamos para imágenes el encoder de antes que ya está entrenado)
attr_encoder = crear_attribute_encoder()

img_encoder = encoder
# Congelamos el experto visual
img_encoder.trainable = False

# ==========================================
#  ENTRENAMIENTO ZERO-SHOT
# ==========================================
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# Usamos nuestra configuración óptima: 7 clases para entrenar
clases_train = [0, 1, 2, 3, 4, 5, 6]

historico_loss = []
historico_acc = []

print("Iniciando entrenamiento Zero-Shot Multi-Modal...")

for episodio in range(1500):
    # Elegimos 5 clases al azar de las 7 disponibles
    c_seleccionadas = np.random.choice(clases_train, size=5, replace=False)

    # Extraemos los vectores de 10 variables exactos para estas 5 clases
    atributos_episodio = matriz_atributos[c_seleccionadas]

    # Recopilamos las imágenes de consulta (15 por clase)
    query_set = []
    for c in c_seleccionadas:
        x_c = x_train_fewshot[y_train_fewshot == c]
        indices_aleatorios = np.random.choice(len(x_c), 15, replace=False)
        query_set.append(x_c[indices_aleatorios])

    query_tensor = tf.convert_to_tensor(query_set, dtype=tf.float32)

    with tf.GradientTape() as tape:
        # Prototipos desde TEXTO/ATRIBUTOS (Pasa de 10 a 256 dimensiones)
        prototipos = attr_encoder(atributos_episodio)

        # Embeddings desde IMÁGENES (Pasa de 32x32x3 a 256 dimensiones)
        query_plano = tf.reshape(query_tensor, [-1, 32, 32, 3])
        z_query = img_encoder(query_plano, training=False)
        #Ponemos training=False porque la red está congelada.
        # Esto es vital para que las capas de BatchNormalization no se rompan.

        # Medir distancias en el espacio de 256 dimensiones
        z_query_exp = tf.expand_dims(z_query, axis=1)
        prototipos_exp = tf.expand_dims(prototipos, axis=0)
        distancias = tf.reduce_sum(tf.square(z_query_exp - prototipos_exp), axis=2)

        # Calcular el error
        logits = -distancias
        etiquetas = tf.repeat(tf.range(5), 15)
        loss = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(etiquetas, logits, from_logits=True))

        # Para el Accuracy: ¿Qué prototipo quedó más cerca? (argmax del logit mayor)
        predicciones = tf.argmax(logits, axis=1)
        # Comparamos predicciones con etiquetas reales (1 si acierta, 0 si falla)
        correctos = tf.cast(tf.equal(predicciones, tf.cast(etiquetas, tf.int64)), tf.float32)
        accuracy = tf.reduce_mean(correctos)

    # SOLO le pedimos a TF los gradientes de la red de atributos
    variables = attr_encoder.trainable_variables
    grads = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(grads, variables))

    # --- GUARDAR Y MOSTRAR HISTORIAL ---
    historico_loss.append(loss.numpy())
    historico_acc.append(accuracy.numpy())

    if (episodio + 1) % 100 == 0:
        media_loss = np.mean(historico_loss[-100:])
        media_acc = np.mean(historico_acc[-100:])
        print(f"Episodios {episodio-99} a {episodio + 1} | Loss Media: {media_loss:.4f} | Accuracy Media: {media_acc*100:.2f}%")

print(" Entrenamiento Zero-Shot completado.")

Iniciando entrenamiento Zero-Shot Multi-Modal...
Episodios 0 a 100 | Loss Media: 1.2357 | Accuracy Media: 53.60%
Episodios 100 a 200 | Loss Media: 0.7710 | Accuracy Media: 72.19%
Episodios 200 a 300 | Loss Media: 0.6932 | Accuracy Media: 73.88%
Episodios 300 a 400 | Loss Media: 0.6528 | Accuracy Media: 75.01%
Episodios 400 a 500 | Loss Media: 0.6434 | Accuracy Media: 75.73%
Episodios 500 a 600 | Loss Media: 0.6422 | Accuracy Media: 74.71%
Episodios 600 a 700 | Loss Media: 0.5773 | Accuracy Media: 78.07%
Episodios 700 a 800 | Loss Media: 0.6118 | Accuracy Media: 75.87%
Episodios 800 a 900 | Loss Media: 0.5945 | Accuracy Media: 77.35%
Episodios 900 a 1000 | Loss Media: 0.6218 | Accuracy Media: 76.91%
Episodios 1000 a 1100 | Loss Media: 0.5984 | Accuracy Media: 77.05%
Episodios 1100 a 1200 | Loss Media: 0.5897 | Accuracy Media: 76.88%
Episodios 1200 a 1300 | Loss Media: 0.5735 | Accuracy Media: 78.17%
Episodios 1300 a 1400 | Loss Media: 0.5984 | Accuracy Media: 77.44%
Episodios 1400 a 150

Tras haber entrenado el modelo, la igual que antes procedemos a evaluarlo con las clases test ejecutando 100 episodios para sacar el promedio

In [ ]:
# Definimos las 3 clases y generamos sus prototipos desde el texto
clases_test = [7, 8, 9]
atributos_test = matriz_atributos[clases_test]

print("Generando los embeddings desde los atributos...")
prototipos_test = attr_encoder(atributos_test) # Esto se calcula 1 sola vez

# Variables para el rigor estadístico
n_episodios_test = 100
n_query = 15
historico_acc_test = []

print(f"Evaluando : Simulando {n_episodios_test} 'episodios' distintos...")

for episodio in range(n_episodios_test):
    query_set_test = []
    etiquetas_test_episodio = []

    # Muestreo aleatorio: En cada episodio sacamos fotos distintas
    for i, c in enumerate(clases_test):
        # Filtramos fotos de esta clase concreta
        x_c = x_test_fewshot[y_test_fewshot == c]
        # Elegimos 15 al azar sin repetir
        indices_aleatorios = np.random.choice(len(x_c), n_query, replace=False)
        query_set_test.append(x_c[indices_aleatorios])

        # Guardamos la etiqueta mapeada (0 para Caballo, 1 para Barco, 2 para Camión)
        etiquetas_test_episodio.extend([i] * n_query)

    query_tensor = tf.convert_to_tensor(query_set_test, dtype=tf.float32)

    # Pasamos las fotos por el experto visual (¡training=False!)
    query_plano = tf.reshape(query_tensor, [-1, 32, 32, 3])
    z_imagenes = img_encoder(query_plano, training=False)

    # Medimos distancias contra los prototipos textuales
    z_imagenes_exp = tf.expand_dims(z_imagenes, axis=1)
    prototipos_test_exp = tf.expand_dims(prototipos_test, axis=0)
    distancias = tf.reduce_sum(tf.square(z_imagenes_exp - prototipos_test_exp), axis=2)

    # Predicción y precisión de ESTE episodio
    predicciones = tf.argmin(distancias, axis=1).numpy()
    correctos = np.sum(predicciones == etiquetas_test_episodio)
    acc_episodio = correctos / len(etiquetas_test_episodio)

    historico_acc_test.append(acc_episodio)

# Estadísticas finales
media_acc = np.mean(historico_acc_test)
desviacion_acc = np.std(historico_acc_test)

print("\n" + "="*60)
print(f"EVALUACION ZERO-SHOT COMPLETADA ({n_episodios_test} EPISODIOS)")
print(f"Precisión Media:       {media_acc * 100:.2f}%")
print(f"Desviación Estándar: ± {desviacion_acc * 100:.2f}%")
print("="*60)

Generando los embeddings desde los atributos...
Evaluando : Simulando 100 'episodios' distintos...

EVALUACION ZERO-SHOT COMPLETADA (100 EPISODIOS)
Precisión Media:       57.11%
Desviación Estándar: ± 3.89%


Hemos obtenido una precisión media de un 57% que quizás puede parecer que no es mucha, pero tenemos que tener en cuenta que el modelo no ha visto **ni un ejemolo de las clases a predecir**, solo un vector de atributos artificial con 10 dimensiones de los elementos de esa clase. Además que si fuera un clasificador trivial como tenemos 3 clases debería de acertar solo un 33%, un 57% es casi el doble.



---

#### Tabla 2: Resultados Zero-Shot Learning

| Modelo | Espacio Semántico (Atributos) | 3-way Zero-Shot |
| :--- | :---: | :---: |
| Puro Azar (Baseline) | - | 33.33% |
| **Prototypical Networks (Nuestra Impl.)** | **Vector Binario (10 dim)** | **57.11% ± 3.89%** |